In [4]:
import numpy as np
from colmap_parsing_utils import read_images_text, write_images_text, Image,read_cameras_text
from numpy.typing import NDArray
from typing import List, Literal, Optional, Tuple
import math
import os
from PIL import Image as PILImage
import shutil

# Setting the dataset path

In [11]:
data_path="dataset/CFGS_Co3d_inter"
scene_list=[ "apple/110_13051_23361" ,"bench/415_57112_110099" , "skateboard/245_26182_52130" ,"hydrant/106_12648_23157" ,]
key_lists=[5,5,5,5,5]

In [5]:
_EPS = np.finfo(float).eps * 4.0
def unit_vector(data: NDArray, axis: Optional[int] = None) -> np.ndarray:
    """Return ndarray normalized by length, i.e. Euclidean norm, along axis.

    Args:
        axis: the axis along which to normalize into unit vector
        out: where to write out the data to. If None, returns a new np ndarray
    """
    data = np.array(data, dtype=np.float64, copy=True)
    if data.ndim == 1:
        data /= math.sqrt(np.dot(data, data))
        return data
    length = np.atleast_1d(np.sum(data * data, axis))
    np.sqrt(length, length)
    if axis is not None:
        length = np.expand_dims(length, axis)
    data /= length
    return data

def quaternion_slerp(
    quat0: NDArray, quat1: NDArray, fraction: float, spin: int = 0, shortestpath: bool = True
) -> np.ndarray:
    """Return spherical linear interpolation between two quaternions.
    Args:
        quat0: first quaternion
        quat1: second quaternion
        fraction: how much to interpolate between quat0 vs quat1 (if 0, closer to quat0; if 1, closer to quat1)
        spin: how much of an additional spin to place on the interpolation
        shortestpath: whether to return the short or long path to rotation
    """
    q0 = unit_vector(quat0[:4])
    q1 = unit_vector(quat1[:4])
    if q0 is None or q1 is None:
        raise ValueError("Input quaternions invalid.")
    if fraction == 0.0:
        return q0
    if fraction == 1.0:
        return q1
    d = np.dot(q0, q1)
    if abs(abs(d) - 1.0) < _EPS:
        return q0
    if shortestpath and d < 0.0:
        # invert rotation
        d = -d
        np.negative(q1, q1)
    angle = math.acos(d) + spin * math.pi
    if abs(angle) < _EPS:
        return q0
    isin = 1.0 / math.sin(angle)
    q0 *= math.sin((1.0 - fraction) * angle) * isin
    q1 *= math.sin(fraction * angle) * isin
    q0 += q1
    return q0

In [9]:
for idx,scene in enumerate(sorted(scene_list)):
    # get camera pose
    print(f"scene:{scene}")
    colmap_images_txt=f"{data_path}/{scene}/sparse/0/images.txt"    # sequential mode's output
    colmap_cameras_txt=f"{data_path}/{scene}/sparse/0/cameras.txt"
    colmap_images_full_txt=f"{data_path}/{scene}/sparse/0/images_full.txt"  # the interpolated camera poses for each images in training set
    training_image_dir = f'{data_path}/{scene}/raw_images'
    images_info=read_images_text(colmap_images_txt) 
    camera_info=read_cameras_text(colmap_cameras_txt)
    '''
    images_info is a dictionart , image_id in colmap : the attribute of images that is namedtuple
    ("id", "qvec", "tvec", "camera_id", "name", "xys", "point3D_ids")
    '''
    # sort the images by name
    images_info = dict(sorted(images_info.items(), key=lambda item: item[1].name))
    for sorted_id, key in enumerate(images_info):
        if (sorted_id == 0):
            start_num = int(images_info[key].name.split('.')[0][-6:])
        elif (sorted_id == len(images_info)-2):
            last_second_num = int(images_info[key].name.split('.')[0][-6:])
        elif (sorted_id == len(images_info)-1):
            end_num = int(images_info[key].name.split('.')[0][-6:])

    # Generate filenames in the range from start_num + 1 to end_num - 1
    filenames = []
    last_section_num = 0
    for filename in sorted(os.listdir(training_image_dir)):
    # Check if the file is a .jpg image
        if filename.endswith('.jpg'):
            filenames.append(filename)
            if(int(filename.split('.')[0][-6:]) >= last_second_num and int(filename.split('.')[0][-6:]) <= end_num):
                last_section_num += 1
    print(len(filenames),filenames)
            
    # interpolation
    quaternions_before = np.array([images_info[colmap_id].qvec for colmap_id in images_info.keys()])
    translation_before = np.array([images_info[colmap_id].tvec for colmap_id in images_info.keys()])
    quaternions_after = []
    translation_after = []
    print(f"quaternions_before {len(quaternions_before)}")
    ts = np.linspace(0, 1, key_lists[idx]+1)   # because key 5

    for i in range(0, len(quaternions_before)-2):
        quaternions_after += [quaternion_slerp(quaternions_before[i], quaternions_before[i+1], t) for t in ts][:-1]
        translation_after += [(1 - t) * translation_before[i] + t * translation_before[i+1] for t in ts][:-1]

    # handcraft the last interpolation
    ts = np.linspace(0, 1, last_section_num)
    print(ts)
    quaternions_after += [quaternion_slerp(quaternions_before[i], quaternions_before[i+1], t) for t in ts]
    translation_after += [(1 - t) * translation_before[i] + t * translation_before[i+1] for t in ts]

    print(f'number of interpolated camera poses: {len(quaternions_after)}')
    # saving
    full_image_info = {}
    # index starts from 1, there is no image_id = 0
    for i in range(1, len(quaternions_after)+1):
        image_id = int(i)
        qvec = quaternions_after[i-1]
        tvec = translation_after[i-1]
        camera_id = 1   # handcraft
        image_name = filenames[i-1]
        xys = np.zeros((1, 2))  # handcraft
        point3D_ids = np.array([-1]) # handcraft
        full_image_info[i] = Image(
            id=image_id,
            qvec=qvec,
            tvec=tvec,
            camera_id=camera_id,
            name=image_name,
            xys=xys,
            point3D_ids=point3D_ids,
        )
    write_images_text(full_image_info, colmap_images_full_txt)


    # resize the images
    # Set the directory containing the original images
    input_dir = f'{data_path}/{scene}/raw_images'
    # Set the directory where the resized images will be saved
    output_dir = f'{data_path}/{scene}/images'
    # Define new size
    new_size = (camera_info[1].width, camera_info[1].height)  # width and height in pixels
    # Iterate over each image in the input directory
    for filename in os.listdir(input_dir):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
            # Construct the full file path
            img_path = os.path.join(input_dir, filename)
            # Open an image file
            with PILImage.open(img_path) as img:
                # Resize the image
                resized_img = img.resize(new_size)
                # Construct the path for the resized image
                output_path = os.path.join(output_dir, filename)
                # Save the resized image
                resized_img.save(output_path)
                # print(f'Resized and saved {filename} to {output_path}')
                
    # # change the reading data
    # shutil.move(f'{data_path}/{scene}/sparse/0/images.txt',f'{data_path}/{scene}/sparse/0/images_raw.txt')
    # shutil.move(f'{data_path}/{scene}/sparse/0/images_full.txt',f'{data_path}/{scene}/sparse/0/images.txt')

scene:teddybear/34_1403_4393
202 ['frame000001.jpg', 'frame000002.jpg', 'frame000003.jpg', 'frame000004.jpg', 'frame000005.jpg', 'frame000006.jpg', 'frame000007.jpg', 'frame000008.jpg', 'frame000009.jpg', 'frame000010.jpg', 'frame000011.jpg', 'frame000012.jpg', 'frame000013.jpg', 'frame000014.jpg', 'frame000015.jpg', 'frame000016.jpg', 'frame000017.jpg', 'frame000018.jpg', 'frame000019.jpg', 'frame000020.jpg', 'frame000021.jpg', 'frame000022.jpg', 'frame000023.jpg', 'frame000024.jpg', 'frame000025.jpg', 'frame000026.jpg', 'frame000027.jpg', 'frame000028.jpg', 'frame000029.jpg', 'frame000030.jpg', 'frame000031.jpg', 'frame000032.jpg', 'frame000033.jpg', 'frame000034.jpg', 'frame000035.jpg', 'frame000036.jpg', 'frame000037.jpg', 'frame000038.jpg', 'frame000039.jpg', 'frame000040.jpg', 'frame000041.jpg', 'frame000042.jpg', 'frame000043.jpg', 'frame000044.jpg', 'frame000045.jpg', 'frame000046.jpg', 'frame000047.jpg', 'frame000048.jpg', 'frame000049.jpg', 'frame000050.jpg', 'frame000051.jpg

In [12]:
for idx,scene in enumerate(sorted(scene_list)):
    # get camera pose
    print(f"scene:{scene}")
    # change the reading data
    shutil.move(f'{data_path}/{scene}/sparse/0/images.txt',f'{data_path}/{scene}/sparse/0/images_raw.txt')
    shutil.move(f'{data_path}/{scene}/sparse/0/images_full.txt',f'{data_path}/{scene}/sparse/0/images.txt')

scene:apple/110_13051_23361
scene:bench/415_57112_110099
scene:hydrant/106_12648_23157
scene:skateboard/245_26182_52130
